In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import jax.numpy as jnp
import numpy as np
import seaborn as snb
from typing import NamedTuple
from typing import Tuple

from scipy.stats import binom as binom_dist
from scipy.stats import beta as beta_dist
from scipy.special import beta as beta_fun

snb.set_style('darkgrid')
snb.set(font_scale=1.5)
plt.rcParams['lines.linewidth'] = 3


# Assignment 1 - 02477 Bayesian Machine Learning

Date: 02/03/2026

## Part 1 - The beta-binomial model

We have the following know parameters:
$$N = 115 \qquad y = 4 \qquad \theta\in [0,1] \qquad  N^* = 20  \qquad   y^* = ?$$
And the models:
$$\theta \approx \text{Beta}(a_0, b_0) \qquad \text{where} \quad a_0=b_0=1$$

$$y|\theta \approx \text{Binomial}(N,\theta)$$

### Task 1.1 Prior mean and 95%-credibility interval
$$\mathbb{E}\left[\theta\right]  = \frac{a_0}{a_0+b_0} = \frac{1}{2}$$

Since a beta distribution with both parameters equal to 1, corresponds to a uniform distribution over the interval [0,1], a 95% credibility interval is just [0.025, 0.975]

### Task 1.2 Posterior mean and 95%-credibility interval

We know that the posterior of theta is another beta distribution given by:
$$ p(\theta|y) = \text{Beta}(\theta|a_0 + y, b_0 + N-y) $$

Therfore the posterior mean is:
$$E_{p(\theta|y)} = \frac{a_0 + y}{b_0 + N-y} = \frac{1 + 4}{1 + 115 - 4} = \frac{5}{112}$$

For the credibility interval, we will use the Beta distribtution defined in exercise 1:

### Task 1.3 Posterior predictive distribution

### Task 1.4 Posterior predictive probability for $N^*=20$

In [1]:
1+2

3

In [ ]:
test_var = 5

### Task 1.5 Mean and variance of the posterior predictive distribution

## Part 2: Linear Gaussian System

### Task 2.1 Determine $p(y)$

We find the marginal distribution $p(y)$ by integrating over $z_1$ and $z_2$ in the joint distribution:
$$p(y) = \int \int p(y, z_1, z_2) \text{d}z_1\text{d}z_2 = \int \int p(y|z_2)p(z_2|z_1)p(z_1) \text{d}z_1\text{d}z_2 $$

We can find this integral by splitting the integration in two parts. First, we integrate over $z_1$, which will give us $z_2$ since $z_2$ does not depend on $y$:
$$p(z_2) = \int p(z_2|z_1)p(z_1) \text{d}z_1 = \int \mathcal{N}(z_1, vI)\mathcal{N}(0, vI) \text{d}z_1$$

Using (3.38) from Murphy1 with proper identification of the means and variances, we get:
$$p(z_2) = \mathcal{N}(z_2 | 0, 2vI)$$

Then we can find the marginal distribution of $y$ as:
$$p(y) = \int p(y|z_2)p(z_2) \text{d}z_2 = \int \mathcal{N}(a^Tz_2, \sigma^2) \mathcal{N}(z_2 | 0, 2vI) \text{d}z_2$$

Again using equation (3.38) we finally get:
$$p(y) = \mathcal{N}(0, \sigma^2 +2va^2)$$


### Task 2.2 Determine $p(y,z_2|z_1)$

We start by using the conditional probability and inserting the expression for the joint probability:
$$p(y, z_2|z_1) = \frac{p(y,z_2, z_1)}{p(z_1)} = \frac{p(y|z_2)p(z_2|z_1)p(z_1)}{p(z_1)} = p(y|z_2)p(z_2|z_1)$$

Therefore $p(y, z_2|z_1)$ can be written as:
$$p(y, z_2|z_1) = \mathcal{N}(a^Tz_2, \sigma^2)\mathcal{N}(z_1, vI)$$

The variance of the product of two multivariate normals with varince $\Sigma_1$ and $\Sigma_2$ is given by:
$$\Sigma_3^{-1} = \Sigma_1^{-1} + \Sigma_1^{-1}$$
and the mean is given by:
$$\mu_3 = \Sigma_3^{-1}(\Sigma_1^{-1}\mu_1 +\Sigma_2^{-1}\mu_2)$$

Therefore:
$$\Sigma_{y,z_2 |z_1} = (\sigma^{-2} + v^{-1}I)^{-1}$$

The mean becomes:
$$\mu_{y,z_2 |z_1} = (\sigma^{-2} + v^{-1}I)(\sigma^{-2}a^Tz_2 + v^{-1}Iz_1) = \frac{z_1 + a^Tz_2}{v\sigma^2} + \frac{z_1}{v^2} + \frac{a^Tz_2}{\sigma^4}$$

So
$$p(y, z_2|z_1) = \mathcal{N}(\frac{z_1 + a^Tz_2}{v\sigma^2} + \frac{z_1}{v^2} + \frac{a^Tz_2}{\sigma^4}, \frac{1}{\sigma^{-2} + v^{-1}I})$$

### Task 2.3 Determine $p(y|z_1)$

Again we start by using the conditional probability:
$$p(y|z_1) = \frac{p(y, z_1)}{p(z_1)}$$

The joint distribution p(y, z_1) can be found by taking integrating the joint distribution over all three variables with regards to $z_2$:
$$p(y|z_1) = \frac{\int p(y, z_2, z_1) \text{d}z_2}{p(z_1)}$$

By inserting and simplifying we arrive at:
$$p(y|z_1) = \frac{\int p(y|z_2)p(z_2|z_1)p(z_1) \text{d}z_2}{p(z_1)} = \frac{p(z_1) \int p(y|z_2)p(z_2|z_1) \text{d}z_2}{p(z_1)} = \int p(y|z_2)p(z_2|z_1) \text{d}z_2$$

If we insert the expressions for the normal distributions and again use (3.38) from Murphy1 we get:
$$p(y|z_1) =\int p(y|z_2)p(z_2|z_1) \text{d}z_2 = \int \mathcal{N}(a^Tz_2,\sigma^2)\mathcal{N}(z_1, vI) \text{d}z_2 = \mathcal{N}(a^Tz_1, \sigma^2 + va^2)$$

### Task 2.4 Determine $p(z_1|y)$

First, we use Bayes rule:
$$p(z_1|y) = \frac{p(y|z_1)p(z_1)}{p(y)}$$
Since we derived both $p(y|z_1)$ and $p(y)$ above, we can write this as:
$$p(z_1|y) = \frac{\mathcal{N}(a^Tz_1, \sigma^2 + va^2)\mathcal{N}(0,vI)}{\mathcal{N}(0,\sigma^2 + 2va^2)}$$

## Part 3 Conjugate model for count data

### Task 3.1 Determine the joint distribution

### Task 3.2 Functional form of a Gamma distribution

### Task 3.3 Posterior distribution $p(\lambda|y)$

### Task 3.4 Posterior distribution for $\lambda$

### Task 3.5 Plot $p(\lambda)$ and $p(\lambda|y)$ for $\lambda\in[0,30]$